In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

# Load the dataset
file_path = '../dataset_for_modeling/features_for_ranking.csv'
rank_teams = pd.read_csv(file_path)

feature_columns = [
    "player_awards", "net_rating", "off_eff", "def_eff", "home_win_pct",
    "away_win_pct", "reb_dif", "conf_win_pct", "ast_dif", "blk_dif",
    "pf_dif", "coach_awards", "efg_pct", "win_pct", "pts_per_poss",
    "ts_pct", "eff_x_pace", "net_four_factors", "pyth_wins"
]

# Add 'rank' and 'confID' for lagging and grouping
base_columns = feature_columns + ['rank', 'confID']

# Ensure data is sorted by team and year for correct lagging
rank_teams = rank_teams.sort_values(by=['tmID', 'year'])


In [37]:
# 1. Create Lag-1 Features (Last Year's Stats)
lag1_feature_names = [f"{col}_lag1" for col in base_columns]
rank_teams[lag1_feature_names] = rank_teams.groupby('tmID')[base_columns].shift(1)

# 2. Create 3-Year Rolling Avg Features
avg3yr_feature_names = [f"{col}_avg3yr_lag" for col in feature_columns]

# Calculate rolling average on the lag-1 features
rolling_avg_features = rank_teams.groupby('tmID')[[f"{col}_lag1" for col in feature_columns]] \
                                 .rolling(window=3, min_periods=1) \
                                 .mean()

# Align the rolling average index
rank_teams[avg3yr_feature_names] = rolling_avg_features.reset_index(level=0, drop=True)

# 3. Clean Data
# Remove all rows with NaN values. This removes:
# a) The first year for every team (no lag-1 data)
# b) The first two years if using min_periods=3 for rolling avg
model_df = rank_teams.dropna(subset=lag1_feature_names + avg3yr_feature_names)

print(f"Original shape: {rank_teams.shape}, Model-ready shape: {model_df.shape}")


Original shape: (142, 64), Model-ready shape: (122, 64)


In [38]:
# The target is always the rank of the CURRENT year
target_col = 'rank'

# Experiment A: Use only last year's stats
features_lag1 = [f"{col}_lag1" for col in feature_columns]

# Experiment B: Use only the 3-year average stats
features_avg3yr = [f"{col}_avg3yr_lag" for col in feature_columns]

# Experiment C: Use both sets
features_combined = features_lag1 + features_avg3yr

feature_sets = {
    "Lag 1 Only": features_lag1,
    "Avg 3-Year Only": features_avg3yr,
    "Combined": features_combined
}


In [39]:
TEST_YEAR = 10
VALIDATION_YEAR = 9

# Train: All years BEFORE the validation year
train_df = model_df[model_df['year'] < VALIDATION_YEAR] # Years 2-8
y_train = train_df[target_col]

# Validate: Only the validation year
val_df = model_df[model_df['year'] == VALIDATION_YEAR] # Year 9
y_val = val_df[target_col]

# Test: Only the test year
test_df = model_df[model_df['year'] == TEST_YEAR] # Year 10
y_test = test_df[target_col]

print(f"Training observations: {len(train_df)}")
print(f"Validation observations: {len(val_df)}")
print(f"Test observations: {len(test_df)}")


Training observations: 96
Validation observations: 13
Test observations: 13


In [40]:
# Get the baseline prediction (last year's rank)
y_pred_baseline = val_df['rank_lag1']

# Evaluate baseline on the Validation set
mae_baseline = mean_absolute_error(y_val, y_pred_baseline)
spearman_baseline, _ = spearmanr(y_val, y_pred_baseline)

print("--- Baseline Model (on Validation Set) ---")
print(f"  MAE: {mae_baseline:.4f}")
print(f"  Spearman Rank: {spearman_baseline:.4f}")


--- Baseline Model (on Validation Set) ---
  MAE: 1.5385
  Spearman Rank: 0.4008


In [41]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []

for fs_name, fs_cols in feature_sets.items():
    print(f"\n--- Testing Feature Set: {fs_name} ---")
    
    # Define X data based on the current feature set
    X_train = train_df[fs_cols]
    X_val = val_df[fs_cols]

    # Scale data (important for Linear Regression)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    for name, model in models.items():
        # Fit model
        if name == 'Linear Regression':
            model.fit(X_train_scaled, y_train)
            y_pred_val = model.predict(X_val_scaled)
        else:
            model.fit(X_train, y_train) # Tree models don't require scaling
            y_pred_val = model.predict(X_val)
        
        # Evaluate model
        mae = mean_absolute_error(y_val, y_pred_val)
        spearman_corr, _ = spearmanr(y_val, y_pred_val)
        
        print(f"  Model: {name}")
        print(f"    MAE: {mae:.4f} (Baseline: {mae_baseline:.4f})")
        print(f"    Spearman: {spearman_corr:.4f} (Baseline: {spearman_baseline:.4f})")
        
        results.append((fs_name, name, mae, spearman_corr))

# Find the best model based on validation Spearman score
best_result = max(results, key=lambda item: item[3])
best_fs_name, best_model_name, best_mae, best_spearman = best_result

print(f"\n--- Best Model from Validation ---")
print(f"  Feature Set: {best_fs_name}")
print(f"  Model: {best_model_name}")
print(f"  Spearman: {best_spearman:.4f}")



--- Testing Feature Set: Lag 1 Only ---
  Model: Linear Regression
    MAE: 1.9985 (Baseline: 1.5385)
    Spearman: 0.0277 (Baseline: 0.4008)
  Model: Random Forest
    MAE: 1.6408 (Baseline: 1.5385)
    Spearman: 0.1440 (Baseline: 0.4008)
  Model: Gradient Boosting
    MAE: 2.0037 (Baseline: 1.5385)
    Spearman: 0.0582 (Baseline: 0.4008)

--- Testing Feature Set: Avg 3-Year Only ---
  Model: Linear Regression
    MAE: 1.7050 (Baseline: 1.5385)
    Spearman: 0.0665 (Baseline: 0.4008)
  Model: Random Forest
    MAE: 1.5908 (Baseline: 1.5385)
    Spearman: 0.1345 (Baseline: 0.4008)
  Model: Gradient Boosting
    MAE: 1.5478 (Baseline: 1.5385)
    Spearman: 0.3823 (Baseline: 0.4008)

--- Testing Feature Set: Combined ---
  Model: Linear Regression
    MAE: 2.0154 (Baseline: 1.5385)
    Spearman: 0.0748 (Baseline: 0.4008)
  Model: Random Forest
    MAE: 1.6446 (Baseline: 1.5385)
    Spearman: 0.3075 (Baseline: 0.4008)
  Model: Gradient Boosting
    MAE: 1.8230 (Baseline: 1.5385)
    Spea

In [42]:
# Get the best model and features
best_model = models[best_model_name]
best_features = feature_sets[best_fs_name]

# Combine Train + Validation data
X_train_full = pd.concat([train_df[best_features], val_df[best_features]])
y_train_full = pd.concat([y_train, y_val])

# Scale the full dataset (if needed)
if best_model_name == 'Linear Regression':
    scaler = StandardScaler()
    X_train_full_scaled = scaler.fit_transform(X_train_full)
    final_model = best_model.fit(X_train_full_scaled, y_train_full)
else:
    final_model = best_model.fit(X_train_full, y_train_full)

print(f"Final model '{best_model_name}' trained on {len(X_train_full)} observations.")


Final model 'Gradient Boosting' trained on 109 observations.


In [43]:
# Prepare the Test set data
X_test = test_df[best_features]
y_test_final = test_df[target_col]

# Scale test data (if needed)
if best_model_name == 'Linear Regression':
    X_test_scaled = scaler.transform(X_test)
    y_pred_final = final_model.predict(X_test_scaled)
else:
    y_pred_final = final_model.predict(X_test)

# Evaluate
mae_final = mean_absolute_error(y_test_final, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test_final, y_pred_final))
spearman_final, _ = spearmanr(y_test_final, y_pred_final)

print("\n--- Final Model Evaluation (on Test Set: Year 10) ---")
print(f"  MAE: {mae_final:.4f}")
print(f"  RMSE: {rmse_final:.4f}")
print(f"  Spearman Rank: {spearman_final:.4f}")



--- Final Model Evaluation (on Test Set: Year 10) ---
  MAE: 1.6789
  RMSE: 2.2934
  Spearman Rank: 0.3435


In [44]:
# Create the final results DataFrame
final_results_df = test_df[['tmID', 'confID', 'year', 'rank', 'rank_lag1']].copy()
final_results_df.rename(columns={'rank': 'actual_rank', 'rank_lag1': 'baseline_rank'}, inplace=True)
final_results_df['predicted_rank_raw'] = y_pred_final

# Rank the model's raw predictions to get a 1-N ranking
final_results_df['predicted_rank_model'] = final_results_df.groupby('confID')['predicted_rank_raw'] \
                                                          .rank(method='first', ascending=True) \
                                                          .astype(int)

# Sort for easy viewing
final_results_df = final_results_df.sort_values(by=['confID', 'actual_rank'])

print("\n--- Final Predicted vs. Actual Rankings (Year 10) ---")
print(final_results_df[['confID', 'tmID', 'actual_rank', 'baseline_rank', 'predicted_rank_model']].to_string(index=False))



--- Final Predicted vs. Actual Rankings (Year 10) ---
confID tmID  actual_rank  baseline_rank  predicted_rank_model
    EA  IND            1            4.0                     1
    EA  ATL            2            7.0                     6
    EA  DET            3            1.0                     2
    EA  WAS            4            6.0                     7
    EA  CHI            5            5.0                     4
    EA  CON            6            2.0                     3
    EA  NYL            7            3.0                     5
    WE  PHO            1            6.0                     5
    WE  SEA            2            2.0                     1
    WE  LAS            3            3.0                     2
    WE  SAS            4            1.0                     4
    WE  MIN            5            7.0                     6
    WE  SAC            6            4.0                     3


In [45]:
# Ideas to improve the modelling:
# - Give weight depending on the year, for example, year 9 is more valuable than year 8 or 7
# - An average of ranks (?) i mean, we can use the rank from the last years, just can't use from the year we are targetting
